# The Slice Filter

*Available as of Qdrant v1.19.0*

The `slice` condition divides a collection into a fixed number of deterministic, disjoint subsets, and matches every point in one of those subsets. Qdrant assigns each point to one slice by hashing its ID, and, for a fixed `total`, slices `0` through `total - 1` are disjoint and together cover every point in the collection.

That determinism is the whole point. Unlike [random sampling](https://qdrant.tech/documentation/search/search/#random-sampling), a given slice always returns the same points, and it composes with any other filter condition. Because `slice` matches on the point ID hash rather than payload data, it needs no payload index: like `has_id`, it is checked per candidate point within each shard that receives the query.

This notebook covers four uses:

- Splitting a collection into disjoint chunks for parallel [scroll](https://qdrant.tech/documentation/concepts/points/#scroll-points), so several workers can export, migrate, or re-embed points without overlapping.
- Restricting a vector search with `query_points` to a fixed, reproducible subset of the collection.
- Drawing a reproducible sample of a collection for recall evaluation or a train/test split.
- Combining `slice` with a payload filter for stratified sampling, for example a canary rollout limited to one category.

### Setup

This notebook runs against a [Qdrant Cloud](https://cloud.qdrant.io) cluster. The [free tier](https://qdrant.tech/documentation/cloud/create-cluster/) is enough, since the collection here is small.

We also use [Qdrant Cloud Inference](https://qdrant.tech/documentation/inference/cloud-inference/) to embed the sample data, so there's no local embedding model to load. Enable Cloud Inference for your cluster from the Inference tab of the Cluster Detail page in the Qdrant Cloud console, where you can also see which models are free to use, marked with a "Cost: Free" label. This tutorial uses `sentence-transformers/all-MiniLM-L6-v2`, one of the free models.

In [ ]:
!pip install -q qdrant-client

Add your cluster URL and API key, both available from the Qdrant Cloud console, as Colab secrets named `QDRANT_URL` and `QDRANT_API_KEY` (the key icon in the left sidebar), then grant this notebook access when prompted. Connect with `cloud_inference=True`.

In [ ]:
from google.colab import userdata
from qdrant_client import AsyncQdrantClient, models

client = AsyncQdrantClient(
    url=userdata.get("QDRANT_URL"),
    api_key=userdata.get("QDRANT_API_KEY"),
    # Without this, inference falls back to running locally.
    cloud_inference=True,
)

collection_name = "slicing-demo"
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

await client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)


Upload a small catalog of product descriptions. Each point's vector is a `Document` object, embedded server-side on Qdrant Cloud.

In [ ]:
import random

from qdrant_client.models import Document

random.seed(0)

products = {
    "electronics": ["wireless earbuds", "4K monitor", "mechanical keyboard", "USB-C hub", "smartwatch"],
    "books": ["science fiction novel", "cookbook", "history book", "graphic novel", "poetry collection"],
    "clothing": ["running shoes", "wool sweater", "denim jacket", "rain jacket", "cotton t-shirt"],
    "home": ["cast iron pan", "ceramic mug", "throw blanket", "desk lamp", "storage basket"],
}
categories = list(products)

points = []
for i in range(500):
    category = random.choice(categories)
    description = random.choice(products[category])
    points.append(
        models.PointStruct(
            id=i,
            payload={"category": category},
            vector=Document(text=description, model=embedding_model),
        )
    )

client.upload_points(collection_name=collection_name, points=points, wait=True)
await client.count(collection_name=collection_name)


CountResult(count=500)

## Scrolling a Single Slice

A `SliceCondition` takes an `index` and a `total`. A single `scroll` call only returns up to `limit` points, so to fetch a whole slice you page through it: keep calling `scroll` with the `next_page_offset` from the previous response until it comes back `None`. Here we page through slice `3` out of `8`, one eighth of the collection.

In [ ]:
next_page_offset = None
records = []
while True:
    result, next_page_offset = await client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(
            must=[models.SliceCondition(slice=models.Slice(index=3, total=8))],
        ),
        limit=500,
        with_payload=False,
        with_vectors=False,
        offset=next_page_offset,
    )
    records.extend(result)
    if next_page_offset is None:
        break

print(f"slice 3 of 8: {len(records)} points")


slice 3 of 8: 61 points


With `total: 8` and 500 points, each slice holds roughly 60 points, close to `500 / 8`.

## Concurrent Scrolling Across Workers

The main use case is splitting a full scroll into `N` independent, non-overlapping scrolls, one per worker. Each worker only needs its own `index`, and since the slices don't overlap, the requests have nothing to coordinate on and can run concurrently. We use `AsyncQdrantClient` and `asyncio.gather` here to fire all four requests at once, rather than waiting on them one at a time.

In [ ]:
import asyncio


async def scroll_slice(index: int, total: int) -> list[models.Record]:
    next_page_offset = None
    records = []
    while True:
        result, next_page_offset = await client.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(
                must=[models.SliceCondition(slice=models.Slice(index=index, total=total))],
            ),
            limit=500,
            with_payload=True,
            with_vectors=False,
            offset=next_page_offset,
        )
        records.extend(result)
        if next_page_offset is None:
            break
    return records


total_slices = 10
slices = await asyncio.gather(*(scroll_slice(i, total_slices) for i in range(total_slices)))

for i, s in enumerate(slices):
    print(f"worker {i}: {len(s)} points")

print(f"sum across workers: {sum(len(s) for s in slices)}")


worker 0: 47 points
worker 1: 56 points
worker 2: 54 points
worker 3: 52 points
worker 4: 41 points
worker 5: 65 points
worker 6: 45 points
worker 7: 45 points
worker 8: 46 points
worker 9: 49 points
sum across workers: 500


Every point appears in exactly one slice, so the counts add up to the full collection with no overlap and no gaps.

## Restricting a Vector Search to a Slice

`slice` is not limited to `scroll`. It also works as a `query_points` filter, so a vector search can be restricted to a fixed, reproducible subset of the collection, for example to hold out a portion of the data for evaluation while searching the rest.

In [ ]:
results = await client.query_points(
    collection_name=collection_name,
    query=models.Document(text="warm winter jacket", model=embedding_model),
    query_filter=models.Filter(
        must=[models.SliceCondition(slice=models.Slice(index=0, total=5))],
    ),
    limit=5,
    with_payload=True,
)


We can now install [`siphash24`](https://pypi.org/project/siphash24/), a python package implementing the same hashing function used for slice filtering in Qdrant, to verify that the retrieved points indeed all belong to slice `0`.

In [ ]:
! pip install siphash24

In [ ]:
from siphash24 import siphash24

def slice_of(point_id: int, total: int) -> int:
    digest = siphash24(point_id.to_bytes(8, "little")).digest()
    return int.from_bytes(digest, "little") % total

print(f"all results in slice 0: {all(slice_of(p.id, 5) == 0 for p in results.points)}")

all results in slice 0: True


## Reproducible Sampling

Because the hash is stable across runs and Qdrant versions, a single slice makes a reproducible sample: requesting slice `0` of `total: 10` always returns the same 10% of the collection. [Random sampling](https://qdrant.tech/documentation/search/search/#random-sampling) cannot make this guarantee, since it draws a fresh random subset on every call, which is why `slice` is the right tool for a recall benchmark or a train/test split that has to be repeatable.

A small helper turns a slice into a set of IDs, so the point can be made in a few lines: calling it twice with the same `index` and `total` returns the exact same IDs.

In [ ]:
async def slice_ids(index: int, total: int) -> set[int]:
    records, _ = await client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(must=[models.SliceCondition(slice=models.Slice(index=index, total=total))]),
        limit=10000,
        with_payload=False,
        with_vectors=False,
    )
    return {p.id for p in records}

first_run = await slice_ids(index=0, total=10)
second_run = await slice_ids(index=0, total=10)

print(f"sample size: {len(first_run)}")
print(f"identical on repeat: {first_run == second_run}")


sample size: 47
identical on repeat: True


Slices with different `total` values are also correlated: slice `0` of `total: 4` is always a subset of slice `0` of `total: 2`. This means you can go from `total: 2` to `total: 4` to halve your sample, and every point in the smaller sample was already in the bigger one, so nothing you have already evaluated drops out.

In [ ]:
coarse_ids = await slice_ids(index=0, total=2)
fine_ids = await slice_ids(index=0, total=4)

print(f"slice 0/4 is a subset of slice 0/2: {fine_ids.issubset(coarse_ids)}")


slice 0/4 is a subset of slice 0/2: True


## Stratified Sampling with Payload Filters

`slice` is a normal filter condition, so it combines with any other condition. This gives a reproducible sample restricted to one category, useful for a canary rollout or for evaluating a change against a single segment of the data.

In [ ]:
from qdrant_client.models import PayloadSchemaType

await client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD,
)

stratified, _ = await client.scroll(
    collection_name=collection_name,
    scroll_filter=models.Filter(
        must=[
            models.SliceCondition(slice=models.Slice(index=0, total=5)),
            models.FieldCondition(key="category", match=models.MatchValue(value="electronics")),
        ],
    ),
    limit=10000,
    with_payload=True,
    with_vectors=False,
)

print(f"electronics points in slice 0/5: {len(stratified)}")
print(f"all match the category: {all(p.payload['category'] == 'electronics' for p in stratified)}")


electronics points in slice 0/5: 28
all match the category: True
